# Lecture 10 Activity – Which vehicles appear in pedestrian deaths?

QSS 20 · Modern Statistical Computing · Fall 2026

## The question

The city's [crash records](https://data.cityofnewyork.us/d/h9gi-nx95)
show 3,469 fatal crashes in the saved data. Its
[persons table](https://data.cityofnewyork.us/d/f55k-p6yu) records 1,805
pedestrians killed. The [vehicles table](https://data.cityofnewyork.us/d/bm4k-52h4)
lists vehicles involved in each crash, but does not identify which vehicle
struck a particular person.

> **In single-vehicle crashes that killed a pedestrian, which vehicle types
> appear, and did their share change between 2012–2018 and 2019–2026?**

A single-vehicle crash links each person to one recorded vehicle. We will
count deaths, not vehicles on the road. These records cannot tell us the risk
of dying near each vehicle type.

### Before you start

- The setup below creates an empty database in memory and loads two saved
  tables. You do not need to choose a database file location.
- `persons` has one row per person killed. `vehicles` has one row per vehicle
  in a fatal crash. `collision_id` links both tables to a crash.
- Run the setup cell once. If `import duckdb` fails, activate `qss20` in a
  terminal and install `python-duckdb` with conda.
- Replace each `...` in a starter cell. Run each query and inspect the result
  before moving on.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option('display.max_rows', 30)

RAW = Path('../data/raw')
con = duckdb.connect()  # an empty, in-memory database

con.execute(f'''
CREATE TABLE persons AS
SELECT CAST(collision_id AS INTEGER) AS collision_id,
       CAST(crash_date AS DATE)      AS crash_date,
       person_type,
       CAST(person_age AS INTEGER)   AS age
FROM read_json_auto('{RAW}/nyc/persons-killed-page-*.json');

CREATE TABLE vehicles AS
SELECT CAST(collision_id AS INTEGER) AS collision_id, vehicle_type
FROM read_json_auto('{RAW}/nyc/fatal-crash-vehicles-batch-*.json');
''')

# Check: 3,636 persons killed and 5,961 vehicles.
con.sql('SELECT COUNT(*) FROM persons').fetchone()[0], \
    con.sql('SELECT COUNT(*) FROM vehicles').fetchone()[0]

(3636, 5961)

## Task 1 · How many vehicles per crash?

Count the vehicles in each crash, then count the crashes by number of
vehicles: one row per number of vehicles, with the number of crashes.

How many fatal crashes involve exactly one vehicle? In a crash with two
vehicles that killed a pedestrian, can these tables tell you which vehicle
struck the pedestrian?

In [2]:
con.sql('''
WITH per_crash AS (
    SELECT collision_id, COUNT(*) AS n_vehicles
    FROM vehicles
    GROUP BY collision_id
)
SELECT n_vehicles, COUNT(*) AS crashes
FROM per_crash
GROUP BY n_vehicles
ORDER BY n_vehicles
''').df()

,n_vehicles,crashes
0,1,1877
1,2,1089
2,3,298
3,4,94
4,5,53
5,6,23
6,7,14
7,8,4
8,9,5
9,10,1


**1,877** fatal crashes involve exactly one vehicle, and 1,089 involve two;
one crash involves 14. With two or more vehicles, the tables cannot say which
one struck a pedestrian: `vehicles` lists every vehicle in the crash, and a
pedestrian's row in `persons` names no vehicle. In a single-vehicle crash, the
one vehicle is the one involved. The rest of the activity keeps those crashes.

## Task 2 · One pedestrian, one vehicle

Build a table `ped_vehicle` with one row per pedestrian killed in a
single-vehicle crash: the pedestrian's `crash_date` and `age`, and the
vehicle's `vehicle_type`.

How many of the 1,805 pedestrians killed remain? What goes wrong if you
count joined rows before restricting the crashes to one vehicle?

In [3]:
con.execute('''
CREATE OR REPLACE TABLE ped_vehicle AS
WITH single AS (
    SELECT collision_id
    FROM vehicles
    GROUP BY collision_id
    HAVING COUNT(*) = 1
)
SELECT p.collision_id, p.crash_date, p.age, v.vehicle_type
FROM persons AS p
JOIN single USING (collision_id)
JOIN vehicles AS v USING (collision_id)
WHERE p.person_type = 'Pedestrian'
''')
con.sql('''
SELECT COUNT(*) AS rows, COUNT(DISTINCT collision_id) AS crashes
FROM ped_vehicle
''').df()

,rows,crashes
0,1482,1473


**1,482** of the 1,805 pedestrians killed died in a crash with exactly one
vehicle, in 1,473 crashes; a few of those crashes killed two pedestrians.
Each pedestrian still has one row, because each single-vehicle crash has one
vehicle. If you count a join to all vehicles before filtering, a pedestrian in a
three-vehicle crash contributes three rows. Those rows name three involved
vehicles, not the vehicle that struck the pedestrian.

## Task 3 · Make four vehicle classes

First inspect the 20 most frequent `vehicle_type` values and their counts.
Then use
`CASE` to make `ped_class` with one row per pedestrian and a new `vclass`:

| Class | Recorded values to include |
|---|---|
| `SUV` | names containing `SPORT UTILITY` or `STATION WAGON` |
| `car` | `SEDAN`, `PASSENGER VEHICLE`, `4 DR SEDAN`, `2 DR SEDAN`, `CONVERTIBLE` |
| `pickup truck` | names containing `PICK` |
| `other or unrecorded` | every remaining value, including missing values |

Use `UPPER(vehicle_type)` so capitals do not change the class. Count the
rows in each class. Which raw spellings did you combine? What does the last
class contain besides missing values?

In [4]:
con.sql('''
SELECT vehicle_type, COUNT(*) AS n
FROM ped_vehicle
GROUP BY vehicle_type
ORDER BY n DESC
LIMIT 20
''').df()

,vehicle_type,n
0,Station Wagon/Sport Utility Vehicle,388
1,Sedan,245
2,PASSENGER VEHICLE,146
3,SPORT UTILITY / STATION WAGON,116
4,Pick-up Truck,53
5,None,52
6,Bus,51
7,BUS,41
8,UNKNOWN,39
9,Dump,35


In [5]:
con.execute('''
CREATE OR REPLACE TABLE ped_class AS
SELECT collision_id, crash_date, age, vehicle_type,
       CASE
           WHEN UPPER(vehicle_type) LIKE '%SPORT UTILITY%'
             OR UPPER(vehicle_type) LIKE '%STATION WAGON%' THEN 'SUV'
           WHEN UPPER(vehicle_type) IN ('SEDAN', 'PASSENGER VEHICLE',
                                        '4 DR SEDAN', '2 DR SEDAN',
                                        'CONVERTIBLE') THEN 'car'
           WHEN UPPER(vehicle_type) LIKE '%PICK%' THEN 'pickup truck'
           ELSE 'other or unrecorded'
       END AS vclass
FROM ped_vehicle
''')
con.sql('''
SELECT vclass, COUNT(*) AS n
FROM ped_class
GROUP BY vclass
ORDER BY n DESC
''').df()

,vclass,n
0,other or unrecorded,512
1,SUV,504
2,car,404
3,pickup truck,62


`Station Wagon/Sport Utility Vehicle` and `SPORT UTILITY / STATION WAGON`
become one SUV class. `Sedan` and `PASSENGER VEHICLE` become one car class.
The remaining class includes buses, vans, trucks, taxis, motorcycles, bikes,
and records with no usable vehicle type. It is not a class of unknown vehicles
only. The four counts must total 1,482, the pedestrian rows kept in Task 2.

## Task 4 · Did the mix change?

Count pedestrian deaths by `vclass` and period. Use `2012-2018` for dates
through 2018 and `2019-2026` for later dates. In pandas, divide each class
count by its period total. Put the periods side by side.

Which of the three named classes gained share? Report both periods' counts
and shares. The first and last calendar years are incomplete, so compare
shares within these saved records, not annual rates.

In [6]:
mix = con.sql('''
SELECT vclass,
       CASE WHEN year(crash_date) <= 2018 THEN '2012-2018'
            ELSE '2019-2026' END AS period,
       COUNT(*) AS n
FROM ped_class
GROUP BY vclass, period
''').df()

mix.groupby('period')['n'].sum()
mix = mix.assign(share=mix['n'] / mix.groupby('period')['n'].transform('sum'))
display(mix.sort_values(['vclass', 'period']).round(3))
shares = mix.pivot(index='vclass', columns='period', values='share')
shares.assign(change=shares['2019-2026'] - shares['2012-2018'])       .sort_values('2019-2026', ascending=False).round(3)

,vclass,period,n,share
5,SUV,2012-2018,237,0.307
7,SUV,2019-2026,267,0.376
4,car,2012-2018,236,0.306
1,car,2019-2026,168,0.237
6,other or unrecorded,2012-2018,276,0.358
0,other or unrecorded,2019-2026,236,0.332
2,pickup truck,2012-2018,23,0.030
3,pickup truck,2019-2026,39,0.055


period,2012-2018,2019-2026,change
vclass,,,
SUV,0.307,0.376,0.069
other or unrecorded,0.358,0.332,-0.025
car,0.306,0.237,-0.069
pickup truck,0.030,0.055,0.025


Of the 772 pedestrian deaths in the first period, 30.7 percent involve an
SUV; among the 710 in the second period, 37.6 percent do. The car share fell
from 30.6 to 23.7 percent. The pickup share rose from 3.0 to 5.5 percent.
The remainder includes several kinds of vehicle and missing types; its share
does not describe one vehicle class.

## Task 5 · What the counts cannot tell you

Write three or four sentences. Do these results show that SUVs are more
dangerous to pedestrians than cars? What would you need to know, and what did
your choices in Tasks 2 and 3 leave out? Name one limit of the date range.

No. The share of pedestrian deaths involving SUVs rose, but SUVs may also
have become a larger share of the vehicles on New York's streets, and these
tables do not count the vehicles on the road or the miles each kind drives. To compare
danger, the deaths per vehicle-mile of each class are needed, from another
source.

Keeping only single-vehicle crashes drops 323 of the 1,805 pedestrian deaths.
Multi-vehicle crashes may involve a different mix. The `other or unrecorded`
class combines several vehicle kinds with missing types. The saved records
also include only part of 2012 and part of 2026. This analysis describes
vehicles recorded in these deaths; it does not explain why the deaths occurred.

## What to take from this

- Decide the row unit of a join before you write it, and restrict the tables
  so that each row means one thing.
- Inspect the recorded vehicle labels before you group them. State which
  labels enter each class and what remains in the catchall class.
- A share of deaths is not a risk. These tables lack vehicle exposure, so
  report which vehicles appear in the selected crashes.